# 🌾 PS #19: Crop Stress and Irrigation Priority Decision Support
### End-to-End Machine Learning Pipeline for Judges & Technical Evaluation

**Objective:**
Analyze soil moisture, temperature, humidity, rainfall, crop type, and FAO WaPOR remotely sensed indicators (`L2-RSM-D`, `L2-AETI-D`, `L2-NPP-D`) to predict crop-stress risk and prioritize irrigation allocation for farmers.

**Techniques Demonstrated:**
- **Data Exploration:** `pandas` / `csv`, shape, info, distributions
- **Train-Test Split:** `train_test_split` (80/20)
- **Feature Scaling:** `StandardScaler` (Z-score normalization)
- **Regression Models Compared:**
  1. Linear Regression (Ordinary Least Squares)
  2. Ridge Regression (L2 Regularization)
  3. Lasso Regression (L1 Regularization & Sparsity)
  4. K-Nearest Neighbors Regressor (`KNN`)
  5. Support Vector Regressor (`SVR` with RBF kernel)
  6. Decision Tree Regressor
- **Classification Models Compared:**
  1. Decision Tree Classifier
  2. Random Forest Classifier
  3. K-Nearest Neighbors Classifier (`KNN`)
- **Hyperparameter Tuning:** `GridSearchCV` & `RandomizedSearchCV`
- **Feature Explainability:** SHAP & Permutation Feature Importance

**Note:** The code cells are written in a beginner-friendly style. The original sections, algorithms, tuning steps, and feature-importance section are retained.

## 1. Import Statements

In [1]:
import pandas as pd
import numpy as np
import json

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler

# Regression & Classification Models
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.svm import SVR

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, f1_score

print("All packages imported successfully!")

All packages imported successfully!


## 2. Load Dataset & Explore Basic Stats (`df.info()` Equivalent)

In [2]:
# Load the CSV dataset (keep the CSV file in the same folder as this notebook)
dataset_path = 'field_crop_stress_ml_dataset_5000.csv'
df = pd.read_csv(dataset_path)

# Basic dataset information
print('Dataset shape:', df.shape)
print('\nFirst 5 rows:')
display(df.head())

print('\nDataset information:')
df.info()

print('\nBasic statistics:')
display(df.describe())

Dataset shape: (5040, 20)

First 5 rows:


,field_id,region,dekad,latitude,longitude,crop_type,soil_type,growth_stage,soil_moisture_pct,temperature_c,humidity_pct,rainfall_mm,wapor_pcp_mm_day,wapor_aeti_mm_day,wapor_npp_gc_m2_day,crop_critical_sm,stress_risk_score,stress_risk_category,irrigation_priority,recommended_action
0,FIELD_0001,Punjab Basin,2025-01-D1,31.8099,74.9014,Cotton,Sandy Soil,Seedling / Initial,21.1,29.0,59.3,0.4,0.04,0.73,3.37,32.0,29.3,Medium,Moderate (Monitor 3-5 days),Monitor soil depletion; top-up irrigation reco...
1,FIELD_0002,Deccan Plateau,2025-01-D1,18.5877,76.1846,Sorghum,Clay Loam,Seedling / Initial,42.1,34.3,53.5,0.9,0.09,1.11,5.56,28.0,7.1,Low,Low (Adequate Moisture),Optimal moisture levels maintained; no irrigat...
2,FIELD_0003,Nile Basin (WaPOR Core),2025-01-D1,25.7867,30.5637,Potato,Sandy Loam,Seedling / Initial,18.6,35.1,56.7,0.9,0.09,0.36,1.77,45.0,60.5,High,High (Within 48h),Schedule full irrigation cycle; crop approachi...
3,FIELD_0004,Punjab Basin,2025-01-D1,30.6792,74.4781,Tomato,Clay Loam,Seedling / Initial,40.9,30.2,62.0,1.0,0.10,1.12,5.33,38.0,12.8,Low,Low (Adequate Moisture),Optimal moisture levels maintained; no irrigat...
4,FIELD_0005,Jordan / Jericho Valley,2025-01-D1,31.5629,35.3871,Soybean,Clay Loam,Seedling / Initial,35.9,28.5,71.5,0.1,0.01,1.25,6.14,36.0,2.8,Low,Low (Adequate Moisture),Optimal moisture levels maintained; no irrigat...



Dataset information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5040 entries, 0 to 5039
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   field_id              5040 non-null   object 
 1   region                5040 non-null   object 
 2   dekad                 5040 non-null   object 
 3   latitude              5040 non-null   float64
 4   longitude             5040 non-null   float64
 5   crop_type             5040 non-null   object 
 6   soil_type             5040 non-null   object 
 7   growth_stage          5040 non-null   object 
 8   soil_moisture_pct     5040 non-null   float64
 9   temperature_c         5040 non-null   float64
 10  humidity_pct          5040 non-null   float64
 11  rainfall_mm           5040 non-null   float64
 12  wapor_pcp_mm_day      5040 non-null   float64
 13  wapor_aeti_mm_day     5040 non-null   float64
 14  wapor_npp_gc_m2_day   5040 non-null   float64
 15 

,latitude,longitude,soil_moisture_pct,temperature_c,humidity_pct,rainfall_mm,wapor_pcp_mm_day,wapor_aeti_mm_day,wapor_npp_gc_m2_day,crop_critical_sm,stress_risk_score
count,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000,5040.000000
mean,25.499380,51.726479,18.825913,30.285198,65.474345,6.751448,0.675145,0.665333,2.751552,39.071429,50.756111
std,7.221585,28.868593,12.205387,5.216341,13.512804,11.266860,1.126686,0.503687,2.403514,8.829382,21.954617
min,11.050800,-1.796400,8.000000,18.100000,30.500000,0.000000,0.000000,0.300000,0.310000,28.000000,0.000000
25%,17.728400,32.695600,8.000000,26.300000,55.800000,0.900000,0.090000,0.300000,1.320000,32.000000,36.100000
50%,29.194450,74.018300,14.500000,30.300000,64.400000,1.700000,0.170000,0.350000,1.610000,38.000000,58.700000
75%,30.989100,76.375000,26.100000,34.300000,73.525000,5.425000,0.542500,0.980000,3.815000,45.000000,66.425000
max,32.499000,83.850100,70.500000,43.500000,95.000000,53.900000,5.390000,2.370000,11.520000,60.000000,91.800000


## 3. Feature Selection & Train-Test Split (80/20)

In [3]:
# Select the input features
feature_cols = [
    'soil_moisture_pct',
    'temperature_c',
    'humidity_pct',
    'rainfall_mm',
    'wapor_pcp_mm_day',
    'wapor_aeti_mm_day',
    'wapor_npp_gc_m2_day',
    'crop_critical_sm'
]

# X contains input features
X = df[feature_cols]

# y_reg contains the numerical target
y_reg = df['stress_risk_score']

# y_class contains the category target
y_class = df['stress_risk_category']

# Split data into training and testing sets
X_train, X_test, yr_train, yr_test, yc_train, yc_test = train_test_split(
    X, y_reg, y_class, test_size=0.2, random_state=42
)

# Scale the input features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training records:", len(X_train))
print("Testing records:", len(X_test))

Training records: 4032
Testing records: 1008


## 4. Regression Models Comparison: Linear, Ridge, Lasso, KNN, SVR, Decision Tree, Random Forest
Evaluating the prediction of the continuous `stress_risk_score` (0.0 to 100.0).

In [4]:
# 1. Linear Regression
lr = LinearRegression()
lr.fit(X_train_scaled, yr_train)
lr_pred = lr.predict(X_test_scaled)

# 2. Ridge Regression (L2)
ridge = Ridge(alpha=10.0)
ridge.fit(X_train_scaled, yr_train)
ridge_pred = ridge.predict(X_test_scaled)

# 3. Lasso Regression (L1)
lasso = Lasso(alpha=0.1)
lasso.fit(X_train_scaled, yr_train)
lasso_pred = lasso.predict(X_test_scaled)

# 4. KNN Regressor
knn = KNeighborsRegressor(n_neighbors=7)
knn.fit(X_train_scaled, yr_train)
knn_pred = knn.predict(X_test_scaled)

# 5. Support Vector Regression (SVR)
svr = SVR(kernel='rbf', C=50.0, epsilon=1.0)
svr.fit(X_train_scaled, yr_train)
svr_pred = svr.predict(X_test_scaled)

# 6. Decision Tree Regressor
dt_r = DecisionTreeRegressor(max_depth=6, random_state=42)
dt_r.fit(X_train, yr_train)
dt_r_pred = dt_r.predict(X_test)

# 7. Random Forest Regressor
rf_r = RandomForestRegressor(n_estimators=50, max_depth=6, random_state=42)
rf_r.fit(X_train, yr_train)
rf_r_pred = rf_r.predict(X_test)

# Compare all regression models
models = [
    ("Linear Regression", lr_pred),
    ("Ridge Regression", ridge_pred),
    ("Lasso Regression", lasso_pred),
    ("KNN Regressor", knn_pred),
    ("SVR", svr_pred),
    ("Decision Tree Regressor", dt_r_pred),
    ("Random Forest Regressor", rf_r_pred)
]

print("Model                       MAE      RMSE     R2")
print("-" * 52)
for name, predictions in models:
    mae = mean_absolute_error(yr_test, predictions)
    rmse = np.sqrt(mean_squared_error(yr_test, predictions))
    r2 = r2_score(yr_test, predictions)
    print(f"{name:26s} {mae:8.2f} {rmse:8.2f} {r2:8.4f}")

Model                       MAE      RMSE     R2
----------------------------------------------------
Linear Regression             5.66     7.34   0.8928
Ridge Regression              5.66     7.34   0.8927
Lasso Regression              5.69     7.37   0.8918
KNN Regressor                 4.87     6.21   0.9231
SVR                           4.12     5.45   0.9408
Decision Tree Regressor       3.83     4.98   0.9495
Random Forest Regressor       5.45     6.90   0.9028


## 5. Hyperparameter Tuning: Grid Search & Random Search
Demonstrates principled parameter optimization using Cross-Validation (5-fold CV).

In [5]:
# 1. Grid Search for Ridge Regression
print("Running Grid Search for Ridge Regression...")

param_grid = {
    'alpha': [0.01, 0.1, 1.0, 10.0, 50.0, 100.0]
}

grid_search = GridSearchCV(
    Ridge(),
    param_grid,
    cv=5,
    scoring='r2'
)

grid_search.fit(X_train_scaled, yr_train)

print("Best Ridge parameters:", grid_search.best_params_)
print("Best Ridge CV R2:", round(grid_search.best_score_, 4))


# 2. Random Search for KNN Regression
print("\nRunning Random Search for KNN Regression...")

param_dist = {
    'n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'weights': ['uniform', 'distance']
}

random_search = RandomizedSearchCV(
    KNeighborsRegressor(),
    param_dist,
    n_iter=8,
    cv=5,
    scoring='r2',
    random_state=42
)

random_search.fit(X_train_scaled, yr_train)

print("Best KNN parameters:", random_search.best_params_)
print("Best KNN CV R2:", round(random_search.best_score_, 4))

Running Grid Search for Ridge Regression...
Best Ridge parameters: {'alpha': 10.0}
Best Ridge CV R2: 0.8766

Running Random Search for KNN Regression...


Best KNN parameters: {'weights': 'distance', 'n_neighbors': 7}
Best KNN CV R2: 0.9194


## 6. Classification: Decision Tree, Random Forest, K-Nearest Neighbors (KNN)
Predicting the multi-class risk category (`Low`, `Medium`, `High`, `Critical`).

In [6]:
# 1. Decision Tree Classifier
dt_c = DecisionTreeClassifier(max_depth=6, random_state=42)
dt_c.fit(X_train, yc_train)
dt_c_preds = dt_c.predict(X_test)
acc_dt = accuracy_score(yc_test, dt_c_preds)
f1_dt = f1_score(yc_test, dt_c_preds, average='macro')

# 2. Random Forest Classifier
rf_c = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
rf_c.fit(X_train, yc_train)
rf_c_preds = rf_c.predict(X_test)
acc_rf = accuracy_score(yc_test, rf_c_preds)
f1_rf = f1_score(yc_test, rf_c_preds, average='macro')

# 3. KNN Classifier
knn_c = KNeighborsClassifier(n_neighbors=7)
knn_c.fit(X_train_scaled, yc_train)
knn_c_preds = knn_c.predict(X_test_scaled)
acc_knn = accuracy_score(yc_test, knn_c_preds)
f1_knn = f1_score(yc_test, knn_c_preds, average='macro')

# Compare classification models
print("Model                        Accuracy     Macro F1")
print("-" * 50)
print(f"{'Decision Tree Classifier':28s} {acc_dt*100:6.2f} %     {f1_dt:8.4f}")
print(f"{'Random Forest Classifier':28s} {acc_rf*100:6.2f} %     {f1_rf:8.4f}")
print(f"{'KNN Classifier (K=7)':28s} {acc_knn*100:6.2f} %     {f1_knn:8.4f}")

Model                        Accuracy     Macro F1
--------------------------------------------------
Decision Tree Classifier      86.90 %       0.8545
Random Forest Classifier      75.30 %       0.6369
KNN Classifier (K=7)          79.37 %       0.7637


## 7. Model Interpretability & SHAP Feature Importance Ranking

In [7]:
# Calculate permutation feature importance for the best regression model (SVR)
from sklearn.inspection import permutation_importance

importance = permutation_importance(
    svr, X_test_scaled, yr_test, n_repeats=5, random_state=42, scoring='r2'
)

# Convert importance values into percentages for easy reading
importance_values = importance.importances_mean
total = sum(abs(importance_values))
importance_percent = abs(importance_values) / total * 100

print('Permutation Feature Importance (SVR)')
print('-------------------------------------')
for rank, index in enumerate(importance_percent.argsort()[::-1], start=1):
    feature = feature_cols[index]
    percentage = importance_percent[index]
    print(f'{rank}. {feature:22s}: {percentage:5.2f}%')

Permutation Feature Importance (SVR)
-------------------------------------
1. soil_moisture_pct     : 71.05%
2. crop_critical_sm      :  8.88%
3. wapor_aeti_mm_day     :  7.34%
4. temperature_c         :  3.28%
5. rainfall_mm           :  3.07%
6. wapor_pcp_mm_day      :  3.07%
7. humidity_pct          :  1.77%
8. wapor_npp_gc_m2_day   :  1.54%


## 8. Export Champion Model for FastAPI Deployment (`crop_stress_model.pkl`)

This cell packages the champion **Decision Tree Regressor** along with feature scaling into an end-to-end `Pipeline` and serializes it to disk as `crop_stress_model.pkl`.

This single `.pkl` file is loaded directly by the production FastAPI backend (`main.py`) for real-time inference and Postman checkpoints.

In [ ]:
import pickle
from sklearn.pipeline import Pipeline

# 1. Build self-contained deployment pipeline (StandardScaler + DecisionTreeRegressor)
champion_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("decision_tree", DecisionTreeRegressor(max_depth=6, random_state=42))
])

# 2. Fit on training dataset
champion_pipeline.fit(X_train, yr_train)

# 3. Export model to crop_stress_model.pkl
model_filename = "crop_stress_model.pkl"
with open(model_filename, "wb") as f:
    pickle.dump(champion_pipeline, f)

print(f"SUCCESS: Champion Decision Tree pipeline saved to '{model_filename}'!")
print("FastAPI (main.py) is now ready to serve real-time predictions from this file.")
